<a href="https://colab.research.google.com/github/mrDevRussia/register/blob/main/Pivot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

# 1. تثبيت المكتبات اللازمة
!pip install flask flask-cloudflared unsloth
from unsloth import FastLanguageModel
from flask import Flask, request, jsonify
from flask_cloudflared import run_with_cloudflared # Changed import

# 2. تحميل الموديل بتاعك (المدمج)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "BedRockC/BedRock-Expert-Full",
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

app = Flask(__name__)

@app.route('/v1/chat', methods=['POST'])
def chat():
    data = request.json
    prompt = data.get("prompt", "")

    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 512, temperature = 0.5)
    response = tokenizer.batch_decode(outputs)[0]

    # تنظيف الرد من البرومبت الأصلي
    clean_response = response.split("Agent Response:")[-1].strip()
    return jsonify({"reply": clean_response})

# 3. تشغيل النفق (Tunnel) والحصول على رابط عام
# Use run_with_cloudflared(app) instead of _run_cloudflared and app.run()
public_url = run_with_cloudflared(app)
print(f"\n[!!!] Your Public API URL: {public_url}/v1/chat")
# app.run() is no longer needed here as run_with_cloudflared handles it


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2

NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [ ]:
!pip install pyngrok

In [ ]:
from flask import Flask, request, jsonify
from transformers import StoppingCriteria, StoppingCriteriaList
import torch
import threading
import re

app = Flask(__name__)

# ─────────────────────────────────────────────────────────────────────────────
#  SYSTEM PROMPT — نفس اللي في pivot.py بالضبط
# ─────────────────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """\
You are Pivot Expert, the absolute authority on the BedRock programming language.
BedRock is a low-level compiled language that targets MIPS-32 big-endian machine code.
It has no runtime, no OS, no GC. Every construct maps to exact machine instructions.

INTENT CLASSIFICATION — Identify the request type FIRST:

[CHAT]     -> Casual conversation, greetings, general questions not about code.
              Respond naturally, like a knowledgeable friend. Keep it short.

[EXPLAIN]  -> "What is X?", "How does Y work?", "Why does Z happen?"
              Give a clear conceptual explanation + relevant code example.

[ANALYZE]  -> "Analyze this code", "What does this do?", "Review my code."
              Deeply analyze the provided BedRock code: logic, bytecode, memory,
              register usage, potential bugs, performance notes.

[DEBUG]    -> "There's a bug", "This doesn't work", "Fix this error."
              Identify the exact bug at parser/lexer/codegen level, explain why
              it fails, and provide the corrected version with notes.

[CREATE]   -> "Write a ...", "Create a ...", "Build a ...", "Generate a ..."
              Create complete, working BedRock code. Always structure as:
              ### Logical Analysis
              [Explain at machine level: memory, registers, bytecode]
              ### Final Code
              ```bedrock
              [complete working code]
              ```

[MODIFY]   -> "Add X to this", "Change Y", "Extend this code", "Refactor."
              Show what changed and why.

[SUGGEST]  -> "How should I approach...", "What's the best way to...", "Recommend..."
              Give architectural advice and trade-offs in BedRock context.

[COMPARE]  -> "Difference between X and Y", "X vs Y"
              Compare clearly with BedRock-specific context.

BEDROCK CORE RULES:
- root declarations -> compile-time constants, emit_li only, never lw
- let declarations  -> static DATA segment address, always sw/lw
- fn bodies         -> prologue: addiu $sp,-32 + sw $ra,28($sp)
- All numbers       -> unsigned 32-bit at codegen
- No floats, no signed types, no implicit casts, no exceptions
- Semicolons        -> MANDATORY after every statement
- if/while          -> condition MUST be in parentheses
- No &&, ||, !, %   -> use nested if / bitwise / manual modulo

RESPONSE LANGUAGE: Match the user language exactly.
DO NOT simulate the next user message. Stop after your answer.\
"""

# ─────────────────────────────────────────────────────────────────────────────
#  STOPPING CRITERIA — وقف الجنريشن فور ما تظهر علامات الهلوسة
# ─────────────────────────────────────────────────────────────────────────────
class MultiTokenStoppingCriteria(StoppingCriteria):
    """
    تقف لما تلاقي أي sequence من الـ stop_sequences في الـ generated tokens.
    بتفحص آخر N tokens في كل خطوة — سريعة ومش بتبطئ الجنريشن.
    """
    def __init__(self, stop_sequences: list[str], tokenizer, device):
        self.stop_ids    = []
        self.tokenizer   = tokenizer
        self.device      = device

        for seq in stop_sequences:
            # encode بدون special tokens عشان نلاقي الـ tokens الفعليين
            ids = tokenizer.encode(seq, add_special_tokens=False)
            self.stop_ids.append(torch.tensor(ids, device=device))

        # أطول sequence — عشان نفحص بس آخر N tokens
        self.max_len = max(len(s) for s in self.stop_ids) if self.stop_ids else 1

    def __call__(self, input_ids: torch.LongTensor, scores, **kwargs) -> bool:
        # فحص آخر max_len tokens فقط
        tail = input_ids[0, -self.max_len:]

        for stop in self.stop_ids:
            n = len(stop)
            if tail.shape[0] >= n:
                if torch.equal(tail[-n:], stop):
                    return True   # وقف
        return False


# ─────────────────────────────────────────────────────────────────────────────
#  PROMPT BUILDER — ChatML format صح لـ Qwen
# ─────────────────────────────────────────────────────────────────────────────
def build_chatml_prompt(messages: list[dict]) -> str:
    """
    messages = [
        {"role": "system",    "content": "..."},
        {"role": "user",      "content": "..."},
        {"role": "assistant", "content": "..."},  # optional history
        ...
    ]
    """
    prompt = ""
    for msg in messages:
        role    = msg["role"]
        content = msg["content"].strip()
        prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"

    # أضف بداية رد الـ assistant — الموديل يكمل من هنا
    prompt += "<|im_start|>assistant\n"
    return prompt


# ─────────────────────────────────────────────────────────────────────────────
#  POST-PROCESSING — تنظيف الـ output
# ─────────────────────────────────────────────────────────────────────────────
def clean_response(raw: str, prompt: str) -> str:
    """
    1. شيل الـ prompt الأصلي من الأول
    2. شيل أي im_end أو im_start tags
    3. لو الموديل بدأ يحاكي المستخدم — اقطع هناك
    4. شيل trailing whitespace
    """
    # ① إزالة الـ prompt
    if raw.startswith(prompt):
        raw = raw[len(prompt):]

    # ② إزالة الـ ChatML tokens اللي ممكن تتسرب
    raw = re.sub(r"<\|im_start\|>.*", "", raw, flags=re.DOTALL)
    raw = re.sub(r"<\|im_end\|>.*",   "", raw, flags=re.DOTALL)

    # ③ لو بدأ يحاكي User turn — اقطع
    #    أنماط مختلفة للـ hallucinated user turns
    cut_patterns = [
        r"\nUser\s*:",
        r"\nuser\s*:",
        r"\n### User",
        r"\n\[User\]",
        r"\nHuman\s*:",
        r"\nQuestion\s*:",
        r"\n<\|im_start\|>user",
    ]
    for pat in cut_patterns:
        m = re.search(pat, raw)
        if m:
            raw = raw[:m.start()]

    # ④ إزالة <eos> و <pad> tokens المكتوبة نصاً
    raw = re.sub(r"<\|endoftext\|>", "", raw)
    raw = re.sub(r"</?s>",           "", raw)

    return raw.strip()


# ─────────────────────────────────────────────────────────────────────────────
#  FLASK ROUTE — /v1/chat
# ─────────────────────────────────────────────────────────────────────────────
@app.route("/v1/chat", methods=["POST"])
def chat():
    data = request.get_json(force=True, silent=True)
    if not data:
        return jsonify({"error": "Invalid JSON body"}), 400

    # ── استقبال الـ messages array (OpenAI format) ────────────────────────────
    messages = data.get("messages", [])

    # Fallback: لو pivot.py القديم بعت "prompt" string بس
    if not messages:
        prompt_text = data.get("prompt", "").strip()
        if not prompt_text:
            return jsonify({"error": "No messages or prompt provided"}), 400
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt_text},
        ]
    else:
        # تأكد إن في system message — لو مش موجود حطه
        has_system = any(m.get("role") == "system" for m in messages)
        if not has_system:
            messages.insert(0, {"role": "system", "content": SYSTEM_PROMPT})
        else:
            # استبدل الـ system message بـ SYSTEM_PROMPT الكامل
            for m in messages:
                if m.get("role") == "system":
                    m["content"] = SYSTEM_PROMPT
                    break

    # ── بناء الـ prompt ────────────────────────────────────────────────────────
    # الأفضل: استخدم apply_chat_template لو متاح لـ Qwen
    try:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,   # يضيف <|im_start|>assistant\n
        )
    except Exception:
        # Fallback: بناء يدوي بـ ChatML
        prompt = build_chatml_prompt(messages)

    # ── Tokenize ───────────────────────────────────────────────────────────────
    inputs = tokenizer(
        [prompt],
        return_tensors="pt",
        truncation=True,
        max_length=3072,          # حد الـ context
    ).to("cuda")

    input_len = inputs["input_ids"].shape[1]

    # ── Stopping Criteria ──────────────────────────────────────────────────────
    stop_sequences = [
        "User:",
        "\nUser:",
        "Human:",
        "\nHuman:",
        "<|im_start|>user",
        "<|im_end|>",             # لما يخلص رده
    ]
    stopping = StoppingCriteriaList([
        MultiTokenStoppingCriteria(stop_sequences, tokenizer, device="cuda")
    ])

    # ── Generation ─────────────────────────────────────────────────────────────
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens      = int(data.get("max_tokens", 1024)),
            temperature         = float(data.get("temperature", 0.3)),
            top_p               = float(data.get("top_p", 0.9)),
            repetition_penalty  = 1.25,        # مهم جداً لمنع التكرار
            do_sample           = True,
            use_cache           = True,
            pad_token_id        = tokenizer.eos_token_id,
            eos_token_id        = tokenizer.eos_token_id,
            stopping_criteria   = stopping,
        )

    # ── Decode — الـ new tokens فقط ────────────────────────────────────────────
    new_ids = output_ids[0][input_len:]        # اقطع الـ prompt
    raw_text = tokenizer.decode(
        new_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    # ── Post-process ───────────────────────────────────────────────────────────
    reply = clean_response(raw_text, prompt)

    # ── رد بـ OpenAI-compatible format ────────────────────────────────────────
    return jsonify({
        "choices": [{
            "message": {
                "role":    "assistant",
                "content": reply,
            },
            "finish_reason": "stop",
            "index": 0,
        }],
        "model": "Pivot-Expert-V1",
    })


# ─────────────────────────────────────────────────────────────────────────────
#  HEALTH CHECK
# ─────────────────────────────────────────────────────────────────────────────
@app.route("/", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": "Pivot-Expert-V1"}), 200

@app.route("/v1/chat", methods=["GET"])
def chat_get():
    return jsonify({"error": "Use POST /v1/chat"}), 405



In [ ]:
from pyngrok import ngrok
import threading

# حط التوكن بتاعك هنا
NGROK_AUTH_TOKEN = "2f5xxOBl4C6Ffh5LhQFo8IECnQh_4csonETo26ksWU66m7uWY"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

def run_flask():
    app.run(host="0.0.0.0", port=5000)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()

public_url = ngrok.connect(5000).public_url
print(f"[+] YOUR PUBLIC API URL: {public_url}/v1/chat")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


[+] YOUR PUBLIC API URL: https://ea55-34-138-36-200.ngrok-free.app/v1/chat
